## Configuration ##

In [2]:
import os
import json
import pathlib
import requests
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from dbrepo.RestClient import RestClient
load_dotenv()

USERNAME = os.getenv("DBREPO_USER")
PASSWORD = os.getenv("DBREPO_PASSWORD")

auth = (USERNAME, PASSWORD)
HOST = "https://test.dbrepo.tuwien.ac.at"
PORT = 3306

API_BASE  = 'https://test.dbrepo.tuwien.ac.at/api/v1'
DATABASE = "data_stewardship_group6_crash_serverity_prediction_lkhb"
DB_ID = "3d81c073-e5fd-49b9-9536-b75ed490ca3e"


print('Configuration loaded.')

Configuration loaded.


In [3]:
import json

def build_table_json(name, description, columns, primary_key_col):
    return {
        "name": name,
        "description": description,
        "columns": columns,
        "constraints": {
            "primary_key": [primary_key_col],
            "foreign_keys": [],
            "uniques": [],
            "checks": []
        },
        "is_public": True,
      "is_schema_public": True
    }

# dtype → dbrepo type
def col(name, col_type, size=None, d=None):
    return {
        "name": name,
        "type": col_type,
        "size": size,
        "d": d,
        "description": None,
        "enums": None,
        "sets": None,
        "index_length": None,
        "null_allowed": False,
        "concept_uri": None,
        "unit_uri": None
    }

collision_json = build_table_json(
    name="collision",
    description="One row per road collision reported to the police in Great Britain during 2023.",
    primary_key_col="collision_index",
    columns=[
        col("collision_index",                              "varchar", size=255),
        col("collision_year",                               "double"),
        col("collision_ref_no",                             "varchar", size=255),
        col("location_easting_osgr",                        "double"),
        col("location_northing_osgr",                       "double"),
        col("longitude",                                    "double"),
        col("latitude",                                     "double"),
        col("police_force",                                 "varchar", size=255),
        col("collision_severity",                           "varchar", size=255),
        col("number_of_vehicles",                           "double"),
        col("number_of_casualties",                         "double"),
        col("date",                                         "varchar", size=50),
        col("day_of_week",                                  "varchar", size=50),
        col("time",                                         "varchar", size=20),
        col("local_authority_district",                     "varchar", size=255),
        col("local_authority_ons_district",                 "varchar", size=255),
        col("local_authority_highway",                      "varchar", size=255),
        col("local_authority_highway_current",              "varchar", size=255),
        col("first_road_class",                             "varchar", size=255),
        col("first_road_number",                            "double"),
        col("road_type",                                    "varchar", size=255),
        col("speed_limit",                                  "double"),
        col("junction_detail",                              "varchar", size=255),
        col("junction_control",                             "varchar", size=255),
        col("second_road_class",                            "varchar", size=255),
        col("second_road_number",                           "double"),
        col("pedestrian_crossing",                          "varchar", size=255),
        col("light_conditions",                             "varchar", size=255),
        col("weather_conditions",                           "varchar", size=255),
        col("road_surface_conditions",                      "varchar", size=255),
        col("special_conditions_at_site",                   "varchar", size=255),
        col("carriageway_hazards",                          "varchar", size=255),
        col("urban_or_rural_area",                          "varchar", size=255),
        col("did_police_officer_attend_scene_of_accident",  "double"),
        col("trunk_road_flag",                              "double"),
        col("lsoa_of_accident_location",                    "varchar", size=255),
        col("enhanced_severity_collision",                  "varchar", size=255),
        col("collision_injury_based",                       "double"),
        col("collision_adjusted_severity_serious",          "double"),
        col("collision_adjusted_severity_slight",           "double"),
    ]
)

vehicle_json = build_table_json(
    name="vehicle",
    description="One row per vehicle involved in a recorded collision.",
    primary_key_col="vehicle_id",
    columns=[
        col("vehicle_id",                       "double"),
        col("collision_index",                  "varchar", size=255),
        col("vehicle_reference",                "double"),
        col("vehicle_type",                     "varchar", size=255),
        col("towing_and_articulation",          "varchar", size=255),
        col("vehicle_manoeuvre",                "varchar", size=255),
        col("vehicle_direction_from",           "varchar", size=255),
        col("vehicle_direction_to",             "varchar", size=255),
        col("vehicle_location_restricted_lane", "varchar", size=255),
        col("junction_location",                "varchar", size=255),
        col("skidding_and_overturning",         "varchar", size=255),
        col("hit_object_in_carriageway",        "varchar", size=255),
        col("vehicle_leaving_carriageway",      "varchar", size=255),
        col("hit_object_off_carriageway",       "varchar", size=255),
        col("first_point_of_impact",            "varchar", size=255),
        col("vehicle_left_hand_drive",          "double"),
        col("journey_purpose_of_driver",        "varchar", size=255),
        col("sex_of_driver",                    "varchar", size=50),
        col("age_of_driver",                    "double"),
        col("age_band_of_driver",               "varchar", size=50),
        col("engine_capacity_cc",               "double"),
        col("propulsion_code",                  "varchar", size=255),
        col("age_of_vehicle",                   "double"),
        col("generic_make_model",               "varchar", size=255),
        col("driver_imd_decile",                "double"),
        col("lsoa_of_driver",                   "varchar", size=255),
        col("escooter_flag",                    "double"),
        col("driver_distance_banding",          "varchar", size=255),
    ]
)

casualty_json = build_table_json(
    name="casualty",
    description="One row per casualty in a recorded collision.",
    primary_key_col="casualty_id",
    columns=[
        col("casualty_id",                          "double"),
        col("collision_index",                      "varchar", size=255),
        col("vehicle_reference",                    "double"),
        col("casualty_reference",                   "double"),
        col("casualty_class",                       "varchar", size=255),
        col("sex_of_casualty",                      "varchar", size=50),
        col("age_of_casualty",                      "double"),
        col("age_band_of_casualty",                 "varchar", size=50),
        col("casualty_severity",                    "varchar", size=255),
        col("pedestrian_location",                  "varchar", size=255),
        col("pedestrian_movement",                  "varchar", size=255),
        col("car_passenger",                        "varchar", size=255),
        col("bus_or_coach_passenger",               "varchar", size=255),
        col("pedestrian_road_maintenance_worker",   "varchar", size=255),
        col("casualty_type",                        "varchar", size=255),
        col("casualty_imd_decile",                  "double"),
        col("lsoa_of_casualty",                     "varchar", size=255),
        col("enhanced_casualty_severity",           "varchar", size=255),
        col("casualty_injury_based",                "double"),
        col("casualty_adjusted_severity_serious",   "double"),
        col("casualty_adjusted_severity_slight",    "double"),
        col("casualty_distance_banding",            "varchar", size=255),
    ]
)

# Send them
import requests

for table_json in [collision_json, vehicle_json, casualty_json]:
    url = f"{HOST}/api/v1/database/{DB_ID}/table"
    response = requests.post(
        url,
        auth=(USERNAME, PASSWORD),
        headers={"Content-Type": "application/json", "Accept": "application/json"},
        json=table_json,
        verify=True
    )
    if response.status_code in (200, 201):
        print(f"{table_json['name']} (id={response.json().get('id')})")
    else:
        print(f"{table_json['name']}: {response.status_code} {response.text}")

collision: 409 {"status":"CONFLICT","message":"Failed to create table: already exists: 409  on POST request for \"http://data-service/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/table\": \"{\"status\":\"CONFLICT\",\"message\":\"Failed to create table: already exists: (conn=1427979) Table 'collision' already exists\",\"code\":\"error.table.exists\"}\"","code":"error.table.exists"}
vehicle: 409 {"status":"CONFLICT","message":"Failed to create table: already exists: 409  on POST request for \"http://data-service/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/table\": \"{\"status\":\"CONFLICT\",\"message\":\"Failed to create table: already exists: (conn=1427981) Table 'vehicle' already exists\",\"code\":\"error.table.exists\"}\"","code":"error.table.exists"}
casualty: 409 {"status":"CONFLICT","message":"Failed to create table: already exists: 409  on POST request for \"http://data-service/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/table\": \"{\"status\":\"CONFLICT\

from dbrepo.RestClient import RestClient
url = f"{API_BASE}/database/{DB_ID}"

client =RestClient(endpoint=HOST, username=USERNAME, password=PASSWORD, secure=True)


for t in tables:
    result = client.create_table(
        database_id=DB_ID,
        name=t['name'],
        is_public=t["is_public"],
        is_schema_public=t["is_schema_public"],
        dataframe=t["dataframe"],
        description=t["description"],
        with_data=False
    )
    print
    print(f"done (table_id={result.id})")

## Create citable identifier ##

In [6]:
import uuid
#This should contain license OGL-UK-3.0 instead of CC-BY-4.0, but DBRepo API wont allow it
payload = {
  "type": "database",
  "titles": [
    {
        "title": "UK Road Safety Open Data 2023",
        "language": "en",
        "type": "Subtitle"
    }
  ],
    "descriptions": [
        {
            "description": (
                "Road safety and traffic collision data for Great Britain for the year 2023, "
                "originally published by the UK Department for Transport under the "
                "Open Government Licence v3.0. "
                "Covers reported accidents, involved vehicles, casualties, "
                "and associated road and environmental conditions. "
                "This relational database was created for academic purposes "
                "as part of the Data Stewardship course at TU Wien (Group 6, 2026)."
            ),
            "language": "en",
            "type": "Abstract"
        }
    ],
    "funders": [
        {
          "funder_name": "Department for Transport, United Kingdom"
        }
    ],
    "licenses": [
        {
            "identifier": "CC-BY-4.0", 
            "uri": "https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/",
            "description": "Open Government Licence v3.0"
        }
    ],
    "publisher": "Crown Copyright – Department for Transport, United Kingdom",
    "language": "en",
    "creators": [
        {
          "affiliation": "Department for Transport, United Kingdom",
          "creator_name": "Department for Transport, United Kingdom",
          "name_type": "Organizational",
          "affiliation_identifier": "https://www.gov.uk/government/organisations/department-for-transport"
        }
      ],
    
      "database_id": DB_ID,
      "publication_year": 2023,
      "related_identifiers": [
        {
            "value": "https://www.gov.uk/government/statistical-data-sets/road-safety-open-data",
            "type": "URL",
            "relation": "IsDerivedFrom"
        }
    ],
}

r = requests.post(
    f"{HOST}/api/v1/identifier",
    auth=(USERNAME, PASSWORD),
    headers={"Content-Type": "application/json", "Accept": "application/json"},
    json=payload,
    verify=True
)
print(r.status_code, r.text)

500 {"timestamp":"2026-05-24T00:06:53.736+00:00","status":500,"error":"Internal Server Error","path":"/api/v1/identifier"}


## Give other members access ##

In [21]:


url = f"{HOST}/api/v1/database/{DB_ID}/access/12549571"

payload = {"type" : "write_all"}

r = requests.post(url, auth=auth, json=payload)

print(r.status_code, r.text)

403 {"status":"FORBIDDEN","message":"Failed to create access to user 12549571: already has access","code":"error.request.forbidden"}
